In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../prepared_data/reduced_vars_with_hmm.csv",
    index_col=0,
    parse_dates=True
)

In [3]:
TARGET_COL = "y_SP500_bin_4w"

y = df[TARGET_COL].copy()

X = df.drop(columns=[TARGET_COL], errors="ignore").copy()
X = X.drop(columns=["SP500"], errors="ignore")  # quita nivel del índice si existe

# Solo numéricas
X = X.select_dtypes(include=[np.number]).copy()

# Limpieza básica
X = X.replace([np.inf, -np.inf], np.nan)

data = X.join(y.rename("target")).dropna()
X = data.drop(columns=["target"])
y = data["target"]

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts(dropna=False))

X shape: (1078, 36)
y value counts:
 target
1.0    857
0.0    221
Name: count, dtype: int64


In [4]:
# Make y numeric if binary as strings (IN/OUT)  -> PIPELINE: IN=0, OUT=1
if y.dtype == "object":
    y_mapped = y.map({"IN": 0, "OUT": 1})
    if y_mapped.isna().any():
        raise ValueError(f"Valores inesperados en y: {y.unique()}")
    y = y_mapped.astype(int)
else:
    # bool -> int, float-int -> int
    if y.dtype == "bool":
        y = y.astype(int)
    elif np.issubdtype(y.dtype, np.number):
        if np.all(np.isclose(y.values, y.values.astype(int))):
            y = y.astype(int)

print("y dtype:", y.dtype, "| unique:", np.unique(y))
print("Counts [IN=0, OUT=1]:", np.bincount(y))

y dtype: int64 | unique: [0 1]
Counts [IN=0, OUT=1]: [221 857]


In [5]:
CUTOFF_DATE = "2024-01-01"  # ajusta si tu compañero usa otra

X_train = X.loc[X.index < CUTOFF_DATE].copy()
X_test  = X.loc[X.index >= CUTOFF_DATE].copy()

y_train = y.loc[y.index < CUTOFF_DATE].copy()
y_test  = y.loc[y.index >= CUTOFF_DATE].copy()

print("Train:", X_train.index.min(), "->", X_train.index.max(), "| n =", len(X_train))
print("Test :", X_test.index.min(), "->", X_test.index.max(), "| n =", len(X_test))
print("y_train counts:\n", y_train.value_counts())
print("y_test counts:\n", y_test.value_counts())

Train: 2005-04-08 00:00:00 -> 2023-12-29 00:00:00 | n = 978
Test : 2024-01-05 00:00:00 -> 2025-11-28 00:00:00 | n = 100
y_train counts:
 target
1    771
0    207
Name: count, dtype: int64
y_test counts:
 target
1    86
0    14
Name: count, dtype: int64


### Balanced Random Forrest

In [6]:
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report, confusion_matrix,
    balanced_accuracy_score
)


In [7]:
tscv = TimeSeriesSplit(n_splits=3)

brf = BalancedRandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

In [8]:
param_grid = {
    "n_estimators": [300, 600],
    "max_depth": [3, 5, 8, None],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5]
}

In [9]:
grid_brf = GridSearchCV(
    estimator=brf,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=tscv,
    n_jobs=-1,
    verbose=1
)

In [10]:
grid_brf.fit(X_train, y_train)

print("Best CV balanced_accuracy:", round(grid_brf.best_score_, 4))
print("Best params:", grid_brf.best_params_)

best_brf = grid_brf.best_estimator_

Fitting 3 folds for each of 48 candidates, totalling 144 fits
Best CV balanced_accuracy: 0.6258
Best params: {'max_depth': 5, 'max_features': 0.5, 'min_samples_leaf': 10, 'n_estimators': 300}


In [11]:
# --- Threshold tuning (optimizando F1 del DOWN) ---
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
    roc_auc_score
)

val_ratio = 0.2
split_val = int(len(X_train) * (1 - val_ratio))

X_tr, X_val = X_train.iloc[:split_val], X_train.iloc[split_val:]
y_tr, y_val = y_train.iloc[:split_val], y_train.iloc[split_val:]

# Entrenar modelo en parte inicial del train
best_brf.fit(X_tr, y_tr)
val_proba = best_brf.predict_proba(X_val)[:, 1]

# Rango de thresholds razonable
thresholds = np.linspace(0.40, 0.55, 31)

best_thr = 0.5
best_score = -1

for thr in thresholds:
    val_pred = (val_proba >= thr).astype(int)

    # optimizamos F1 del DOWN (label 0)
    f1 = f1_score(y_val, val_pred, pos_label=0)
    if f1 > best_score:
        best_score = f1
        best_thr = thr

print(f"Best threshold (val): {best_thr:.3f} | Best F1 IN (val): {best_score:.4f}")

# Reentrenar modelo con TODO el train
best_brf.fit(X_train, y_train)

# Probabilidades en test
y_proba = best_brf.predict_proba(X_test)[:, 1]

# Aplicar threshold óptimo
y_pred = (y_proba >= best_thr).astype(int)

# -----------------------------
# STEP — Test analytics (BINARY) — prettier confusion matrix
# -----------------------------
acc = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

# Match the style of your main model prints
print("\n=== Balanced Random Forest (TEST) ===")
print("Accuracy:", round(acc, 4))
print("Balanced Acc:", round(bacc, 4))
print("ROC-AUC:", round(auc, 4))

# Confusion matrix as labeled DataFrame (same style as main model)
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["True OUT (0)", "True IN (1)"],
    columns=["Pred OUT (0)", "Pred IN (1)"]
)

print("\nConfusion Matrix:")
print(cm_df)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["OUT (0)", "IN (1)"]))

Best threshold (val): 0.450 | Best F1 IN (val): 0.4894

=== Balanced Random Forest (TEST) ===
Accuracy: 0.89
Balanced Acc: 0.7566
ROC-AUC: 0.8887

Confusion Matrix:
              Pred OUT (0)  Pred IN (1)
True OUT (0)             8            6
True IN (1)              5           81

Classification Report:
              precision    recall  f1-score   support

     OUT (0)       0.62      0.57      0.59        14
      IN (1)       0.93      0.94      0.94        86

    accuracy                           0.89       100
   macro avg       0.77      0.76      0.76       100
weighted avg       0.89      0.89      0.89       100



#### Metrics

In [12]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# --- TRAIN split (fit set)
ptr = best_brf.predict_proba(X_tr)[:, 1]
pred_tr = (ptr >= best_thr).astype(int)

# --- VAL split (threshold-tuning set)
pva = val_proba  # already computed above
pred_va = (pva >= best_thr).astype(int)

# --- TEST
pte = y_proba    # already computed above
pred_te = (pte >= best_thr).astype(int)

def r4(x):
    return float(f"{x:.4f}")

print(f"\n=== QUICK METRICS (0=IN, 1=OUT) @ thr={best_thr:.3f} ===")
print("TRAIN  acc:", r4(accuracy_score(y_tr, pred_tr)),
      "bal_acc:", r4(balanced_accuracy_score(y_tr, pred_tr)),
      "F1_OUT:", r4(f1_score(y_tr, pred_tr, pos_label=1)))

print("VAL    acc:", r4(accuracy_score(y_val, pred_va)),
      "bal_acc:", r4(balanced_accuracy_score(y_val, pred_va)),
      "F1_OUT:", r4(f1_score(y_val, pred_va, pos_label=1)))

print("TEST   acc:", r4(accuracy_score(y_test, pred_te)),
      "bal_acc:", r4(balanced_accuracy_score(y_test, pred_te)),
      "F1_OUT:", r4(f1_score(y_test, pred_te, pos_label=1)))


=== QUICK METRICS (0=IN, 1=OUT) @ thr=0.450 ===
TRAIN  acc: 0.8747 bal_acc: 0.8445 F1_OUT: 0.9183
VAL    acc: 0.7551 bal_acc: 0.693 F1_OUT: 0.8389
TEST   acc: 0.89 bal_acc: 0.7566 F1_OUT: 0.9364


In [13]:
# =========================
# Balanced Random Forest feature importance (THIS run: best_brf)
# Uses impurity-based importance (Gini decrease)
# =========================

# Safety: some wrappers expose importances after fit; ensure best_brf is fitted
importances = getattr(best_brf, "feature_importances_", None)

if importances is None:
    print("This model does not expose feature_importances_.")
else:
    imp_df = (
        pd.DataFrame({"feature": X_train.columns, "importance": importances})
          .sort_values("importance", ascending=False)
          .reset_index(drop=True)
    )

    print("Total features with non-zero importance:", int((imp_df["importance"] > 0).sum()))
    display(imp_df.head(40))  # top 40

Total features with non-zero importance: 36


,feature,importance
0,NFCI__d1,0.181960
1,NFCI__d1__z8,0.156450
2,NFCI__lvl__z8,0.056939
3,CPI__pct52,0.045471
4,US_2Y_Treasury__lvl__rollstd8,0.038596
5,WTI_Crude_Oil__pct52,0.033881
6,Building_Permits__pct1__rollstd8,0.030310
7,US_10Y_Treasury__lvl__rollstd52,0.027741
8,Natural_Gas__pct52,0.025591
9,US_3M_Treasury__d1,0.024615


### Data extraction

In [14]:
# =====================================================
# DF for trading sim (2024+ only) + simple checks + save
# Model: Balanced Random Forest (best_brf)
# =====================================================

# Build df (2024+)
df_trading = df.loc[df.index >= CUTOFF_DATE].copy()

# Sanity: X_test index must match df_trading index (same dates, same order)
if not df_trading.index.equals(X_test.index):
    print("WARNING: df_trading.index != X_test.index")
    print("df_trading:", df_trading.index.min(), "->", df_trading.index.max(), "n=", len(df_trading))
    print("X_test    :", X_test.index.min(),     "->", X_test.index.max(),     "n=", len(X_test))
    missing_in_df = X_test.index.difference(df_trading.index)
    missing_in_X  = df_trading.index.difference(X_test.index)
    print("Missing in df_trading (should be 0):", len(missing_in_df))
    print("Missing in X_test (should be 0):", len(missing_in_X))
    if len(missing_in_df) > 0: print("Example missing_in_df:", missing_in_df[:5].tolist())
    if len(missing_in_X)  > 0: print("Example missing_in_X :", missing_in_X[:5].tolist())

# Add preds (aligned by index)
# NOTE:
# - y_proba: P(OUT=1) from best_brf.predict_proba(X_test)[:, 1]
# - y_pred : (y_proba >= best_thr).astype(int)
df_trading["p_out"] = pd.Series(y_proba, index=X_test.index)
df_trading["pred_out"] = pd.Series(y_pred, index=X_test.index)
df_trading["y_out_true"] = pd.Series(y_test, index=X_test.index)

# NaN checks (just the important columns)
nan_counts = df_trading[["p_out", "pred_out", "y_out_true"]].isna().sum()
print("\nNaNs in key cols:\n", nan_counts)

# quick assertion-like prints
print("\nRows in df_trading:", len(df_trading))
print("Pred rows (X_test):", len(X_test))
print("All key cols non-null? ->", (nan_counts.sum() == 0))

# Save
out_path = "../predictions/brf_preds.csv"
df_trading.to_csv(out_path)
print("\nSaved ->", out_path)


NaNs in key cols:
 p_out         0
pred_out      0
y_out_true    0
dtype: int64

Rows in df_trading: 100
Pred rows (X_test): 100
All key cols non-null? -> True

Saved -> ../predictions/brf_preds.csv
